# 리포트 46 — 여섯 항목은 닫힌형이고, 점유 대가만 몬테카를로 격자에서 읽는다

> ### 한 일
> **조명원 선택이 무는 dB 격차를 항목별로 모아 원장을 만들고, 항목마다 그 값을 닫는 방식이 무엇인지를 함께 적었다.**

### 결과
1. 원장 항목은 전부 **같은 표적·같은 기하에서 잰 두 양의 비**라 표적 σ 가 분자와 분모에서 상쇄된다 — 반송파 $\lambda^2$ -9.03 dB [^1] (밴드 양끝) · WiFi 패킷 듀티 -12.84 dB [^2] · CPI 규약 3.01 dB [^3].
2. 여섯 항목은 자원격자 · 반송파 · 관측시간에서 **닫힌형**으로 나온다.
3. 점유 대가만 검출 몬테카를로의 EIRP 격자에서 **읽은** 값이다 — 격자점 차 18 dB [^4], 참값 구간 12 [^5]~24 dB [^6], $P_d$ 선형보간 16.4 dB [^7].
4. 그 격자는 눈금 6 dB [^8] · 시행 60 [^9]회 · 표적 mavic4pro [^10] · radial [^11] 한 점에서 읽었다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 항목의 형태 | 전부 두 양의 비다 — $\lambda^2$ 는 반송파 비, 듀티는 시간 비, 기준신호 에너지는 자원격자 위의 에너지 비 |
| 점유 대가 | 같은 표적·같은 기하에서 $P_d$ 0.5 [^12] 를 넘기는 EIRP 차를 6 dB [^8] 격자에서 읽는다 |
| 부호 규약 | 표의 부호는 원본 JSON 그대로이고, 그림은 **음수 = 손해**로 부호를 맞춰 다시 그린 것이다 |
| 점유 대가 안에 든 것 | G1→G3 은 점유율과 함께 기준신호 대역도 7.2 MHz [^13] → 98.28 MHz [^14] 로 넓어진 값이다 — 두 항을 가르는 대역고정 스윕은 다음 단계에 있다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
~/.venvs/py312/bin/python src/viz_report2.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/report4_fixups.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/report2_waveform_rcs.json`, `outputs/report4_fixups.json`, `outputs/report5_results.json`, `outputs/report03_illuminators.json` |
| 소요 | ③ 556 s [^15] · ④ CPU 20초 안쪽 |
| 비고 | `outputs/report5_results.json` 는 검출 몬테카를로가 이미 남긴 것이다 — 이 편은 그중 `A_occupancy` 만 인용한다(재실행 불필요). |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 45 «5G 는 좁고 드물다»](45_5g-double-cost.ipynb) | 5G 가 거리·속도 두 축에서 무는 대가 |

---

## 원장 — 무엇이 각 항목의 유효숫자를 정하나

조명원 선택이 만드는 dB 격차를 한 표에 모은다. 오른쪽 열이 그 항목을 **닫는 방식**이다.

| 항목 | 값 | 무엇의 비인가 | 닫는 방식 |
|---|---|---|---|
| 점유 대가 (5G · 상시 vs 풀로드) | 18 dB [^4] | 같은 표적·같은 기하에서 $P_d$ 0.5 [^12] 를 넘기는 EIRP 차 | 몬테카를로 격자 읽기 |
| 기준신호 에너지 격차 (같은 쌍) | 12.70 dB [^16] | $E_{ref}$(G3) / $E_{ref}$(G1) — 상관에 쓰는 에너지만 | 닫힌형 — 자원격자 |
| 반송파 $\lambda^2$ (LTE→WiFi) | -9.03 dB [^17] | $20\log_{10}(\lambda/\lambda_{ref})$ — EIRP·수신이득 고정 | 닫힌형 — 반송파 |
| 반송파 $\lambda^2$ (LTE→5G) | -5.57 dB [^18] | 위와 같음 | 닫힌형 — 반송파 |
| WiFi 파일럿 / 총 송신 에너지 | -11.27 dB [^19] | G3 격자에서 상관에 쓰는 몫 | 닫힌형 — 자원격자 |
| WiFi 패킷 듀티 | -12.84 dB [^2] | 패킷이 공중에 있는 시간 비율 | 닫힌형 — 시간 |
| CPI 규약 격차 | 3.01 dB [^3] | 같은 프레임 수 M 이 5G 에 주는 관측시간이 절반 | 닫힌형 — 관측시간 |

## 각 항목이 서는 조건

| 항목 | 성립 조건 | 크기 |
|---|---|---|
| 반송파 $\lambda^2$ | EIRP 고정 · 수신 안테나 **이득** 고정 (`src/freespace_link.py:371`) | 수신 **개구면적**을 고정하면 부호가 뒤집힌다 |
| CPI 규약 | 같은 M 이 5G 에 주는 관측시간이 절반 | 3.01 dB [^3] — 뒤 편들은 관측시간을 맞춘 뒤 비교한다 |
| WiFi 두 항목 | 에너지 비 · 시간 비 — 서로 다른 양이다 | -11.27 dB [^19] · -12.84 dB [^2] |

## 점유 대가는 왜 격자에서 읽는가

이 항목만 닫힌형이 없다. 같은 표적·같은 기하에서 $P_d$ 0.5 [^12] 를 넘기는 EIRP 를 상시 체제와 풀로드 체제에서 각각 찾아 그 차를 읽는다. 격자 눈금이 유효숫자를 정한다.

| 무엇 | 값 |
|---|---|
| 격자점 차 | 18 dB [^4] |
| 참값 구간 | 12 [^5]~24 dB [^6] |
| $P_d$ 선형보간 | 16.4 dB [^7] |
| 격자 눈금 · 시행 | 6 dB [^8] · 60 [^9]회 |
| 기준신호 대역 (G1 → G3) | 7.2 MHz [^13] → 98.28 MHz [^14] |

![report03_f3_occupancy](../outputs/figures/report03_f3_occupancy.png)

**그림 1.** 셀이 데이터로 바빠지면 패시브의 거리분해능도 같이 좋아지는가?

![report03_f4_ledger](../outputs/figures/report03_f4_ledger.png)

**그림 2.** 조명원 선택이 무는 대가는 항목별로 몇 dB 인가?

## 이 원장이 σ 논의와 독립인 이유

항목마다 분자와 분모가 같은 표적·같은 기하에서 나온다. 그래서 표적 σ 의 절대레벨이 X dB 움직여도 세 조명원의 **순위와 격차는 그대로**이고, 움직이는 것은 절대 검출거리뿐이다.

검출 결과 편들이 여기에 σ 와 기하를 곱해 절대 거리를 낸다 — [편 57 «세 밴드에서 값이 다른 항은 λ² 와 σ 둘뿐이다»](57_sensitivity-chain.ipynb).

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| EIRP 격자를 6 dB [^8] 에서 2 dB 로 좁히고 기준신호 대역을 고정한 점유 스윕을 돌린다 | 18.0 dB [^4] 안에서 점유 항과 대역 항의 크기가 갈린다 | `benchmark/run_matrix.py:300` |
| 표적 mavic4pro [^10] · 시나리오 radial [^11] 한 점에서 읽은 점유 대가를 기체·기하로 넓힌다 | 점유 대가가 표적·기하에 얼마나 의존하는지가 수치로 확정된다 | [편 60 «앵커 σ 위의 R90 은 3.69~11.10…»](60_r90.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 19개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report03_illuminators.json` | `lambda2.span_db` | -9.026 |
| [^2] | `outputs/report4_fixups.json` | `F4_linkbudget.wifi_pilot_fraction.packet_duty_db` | -12.84 |
| [^3] | `outputs/report4_fixups.json` | `F4_linkbudget.cpi_asymmetry.span_db` | 3.01 |
| [^4] | `outputs/report03_illuminators.json` | `occupancy_cost.value_db` | 18 |
| [^5] | `outputs/report03_illuminators.json` | `occupancy_cost.bracket_lo_db` | 12 |
| [^6] | `outputs/report03_illuminators.json` | `occupancy_cost.bracket_hi_db` | 24 |
| [^7] | `outputs/report03_illuminators.json` | `occupancy_cost.interp_db` | 16.41 |
| [^8] | `outputs/report03_illuminators.json` | `occupancy_cost.eirp_grid_step_db` | 6 |
| [^9] | `outputs/report03_illuminators.json` | `occupancy_cost.n_trials` | 60 |
| [^10] | `outputs/report03_illuminators.json` | `occupancy_cost.drone` | mavic4pro |
| [^11] | `outputs/report03_illuminators.json` | `occupancy_cost.scen` | radial |
| [^12] | `outputs/report03_illuminators.json` | `occupancy_cost.pd_threshold` | 0.5 |
| [^13] | `outputs/report03_illuminators.json` | `occupancy_cost.ref_bw_G1_mhz` | 7.2 |
| [^14] | `outputs/report03_illuminators.json` | `occupancy_cost.ref_bw_G3_mhz` | 98.28 |
| [^15] | `outputs/report4_fixups.json` | `_meta.runtime_s` | 555.5 |
| [^16] | `outputs/report03_illuminators.json` | `ref_energy_gap_G1_to_G3_db.nr` | 12.7 |
| [^17] | `outputs/report03_illuminators.json` | `lambda2.lte_to_wifi_db` | -9.026 |
| [^18] | `outputs/report03_illuminators.json` | `lambda2.lte_to_nr_db` | -5.571 |
| [^19] | `outputs/report4_fixups.json` | `F4_linkbudget.wifi_pilot_fraction.pilot_over_tx_energy_db` | -11.27 |